# Anotación asistida con CVAT (pre-anotación YOLO26 + revisión humana)

**Por qué existe esta fase.** Si las etiquetas de entrenamiento se generan con el mismo detector preentrenado que luego se ajusta, el mAP solo mide cuánto se parece el modelo a su "profesor", no a la realidad (entrenamiento circular). Aquí las cajas las propone YOLO26, pero **un humano las revisa y corrige en CVAT**, y solo las etiquetas revisadas pasan a entrenamiento y test.

**Encaje en el pipeline:**

```
02_preprocess_labelled_*  ──► data/malaga/fases/{frames,weather}.parquet  (fotogramas válidos + contexto)
        │
03_anotacion_cvat (este)   ──► 1. muestreo estratificado (cámara × franja × tipo de día × iluminación)
        │                      2. tarea CVAT + pre-anotación automática YOLO26 (cvat_sdk.auto_annotation)
        │                      3. revisión humana en http://localhost:8080
        │                      4. exportación ──► data/malaga/anotacion/<lote>/labels/*.txt
        │                                         data/malaga/anotacion/<lote>/indice.csv
        ▼
04_training                ──► lee todos los anotacion/*/indice.csv (solo estado = revisada)
```

Las clases están definidas en `src/deteccion.py` (`CLASS_NAMES`), compartidas con `04_training`. La función de pre-anotación está en `src/cvat_yolo26.py`.

**Requisitos:** Docker Desktop en ejecución y CVAT arrancado:

```powershell
.\cvat\cvat.ps1 instalar   # solo la primera vez (clona CVAT v2.78.0, arranca y crea el usuario admin)
.\cvat\cvat.ps1 arrancar   # sesiones posteriores
```

## 1. Configuración

In [ ]:
import json
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from anotacion import find_label_files, read_export_names, remap_lines
from config import load_config, path
from deteccion import CLASS_NAMES, CLASS_TO_IDX
from pipeline import image_path, load_frames, stratified_selection

CFG = load_config()
ANOTACION_DIR = path("anotacion")

# Lote de anotación (config.yaml -> cvat): cada lote nuevo excluye las imágenes de lotes anteriores
LOTE = CFG["cvat"]["lote"]
N_IMAGENES_LOTE = CFG["cvat"]["n_imagenes_lote"]
MAX_POR_CAMARA = CFG["cvat"]["max_por_camara"]     # evita que unas pocas cámaras dominen el lote
SEGMENT_SIZE = CFG["cvat"]["segment_size"]         # imágenes por job de CVAT (unidad de revisión)
RANDOM_STATE = CFG["entrenamiento"]["semilla"]

# Pre-anotación YOLO26 (mismos parámetros que 02_preprocess_labelled_detection)
PRE_WEIGHTS = CFG["deteccion"]["modelo"]
PRE_CONF = CFG["deteccion"]["conf"]
PRE_IMGSZ = CFG["deteccion"]["imgsz"]

CVAT_PROJECT_NAME = CFG["cvat"]["proyecto"]

# Formato de exportación (se comprueba que el servidor lo soporte)
EXPORT_FORMATS = ["Ultralytics YOLO Detection 1.0", "YOLO 1.1"]

LOTE_DIR = ANOTACION_DIR / LOTE
INDICE_FILE = LOTE_DIR / "indice.csv"
TASK_FILE = LOTE_DIR / "cvat_task.json"

# Credenciales generadas por cvat\cvat.ps1 (cvat\.env, excluido de git)
def read_env(path):
    values = {}
    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip()
    return values

CVAT = read_env(PROJECT_ROOT / "cvat" / ".env")
print("CVAT:", CVAT["CVAT_URL"], "| usuario:", CVAT["CVAT_USER"])
print("Clases:", CLASS_NAMES)
print("Lote:", LOTE_DIR)

## 2. Conexión con CVAT

La versión del servidor debe coincidir con `cvat-sdk` (fijada en `requirements.txt` y en `cvat/cvat.ps1`).

In [ ]:
import cvat_sdk
from cvat_sdk import make_client

client = make_client(CVAT["CVAT_URL"], credentials=(CVAT["CVAT_USER"], CVAT["CVAT_PASSWORD"]))
server_version = client.api_client.server_api.retrieve_about()[0].version
print(f"Servidor CVAT {server_version} | cvat-sdk {cvat_sdk.__version__}")

if server_version.lstrip("v").split("-")[0] != cvat_sdk.__version__:
    print("Aviso: versiones distintas de servidor y SDK; pueden fallar algunas llamadas.")

formats = client.api_client.server_api.retrieve_annotation_formats()[0]
exporters = {f.name for f in formats.exporters}
EXPORT_FORMAT = next(f for f in EXPORT_FORMATS if f in exporters)
print("Formato de exportación:", EXPORT_FORMAT)

## 3. Selección estratificada de fotogramas

Se parte de los fotogramas válidos de la fase `frames` (unida con `weather` si existe) y se reparte el lote en *round-robin* (`pipeline.stratified_selection`) entre estratos **cámara × franja horaria × tipo de día × iluminación**, con un máximo por cámara. Así el lote cubre día/noche, laborables/fines de semana y el mayor número de cámaras posible, en lugar de concentrarse en las horas o cámaras con más capturas.

Se excluyen las imágenes ya incluidas en lotes anteriores. Las imágenes se copian con nombre único `<camara>_<timestamp>.jpg` (los timestamps se repiten entre cámaras).

In [ ]:
frames = load_frames("weather")
frames["source_path"] = frames["path"].map(image_path)
frames = frames[frames["source_path"].map(Path.exists)]
frames["image"] = frames["camera"] + "_" + frames["image"]

# Imágenes ya anotadas o pendientes en otros lotes
previous = [
    pd.read_csv(f, dtype={"camera": str}) for f in ANOTACION_DIR.glob("*/indice.csv") if f.parent != LOTE_DIR
]
already = set(pd.concat(previous)["image"]) if previous else set()
candidates = frames[~frames["image"].isin(already)]
print(f"Fotogramas disponibles: {len(frames)} | ya en otros lotes: {len(already)} | candidatos: {len(candidates)}")

strata = [c for c in ["camera", "time_slot_2h", "day_type", "illumination"] if c in candidates.columns]
lote = stratified_selection(candidates, strata, N_IMAGENES_LOTE, MAX_POR_CAMARA, RANDOM_STATE)

print(f"Imágenes seleccionadas: {len(lote)} de {lote['camera'].nunique()} cámaras")
for col in ["time_slot_2h", "day_type", "illumination", "weather"]:
    if col in lote.columns:
        print(f"\n{col}:\n{lote[col].value_counts().sort_index().to_string()}")

In [ ]:
images_dir = LOTE_DIR / "images"
if INDICE_FILE.exists():
    raise FileExistsError(f"El lote {LOTE} ya existe ({INDICE_FILE}). Cambia LOTE para crear uno nuevo.")

images_dir.mkdir(parents=True, exist_ok=True)
for row in lote.itertuples():
    shutil.copy2(row.source_path, images_dir / row.image)

keep = [c for c in ["image", "camera", "timestamp", "timestamp_local", "time_slot_2h", "day_type", "illumination", "weather"] if c in lote.columns]
indice = lote[keep].copy()
indice["source_path"] = lote["path"].values
indice["lote"] = LOTE
indice["label_path"] = ""
indice["n_cajas"] = pd.NA
indice["estado"] = "pendiente"
indice.to_csv(INDICE_FILE, index=False)
print(f"Copiadas {len(indice)} imágenes a {images_dir}")
print(f"Índice del lote: {INDICE_FILE}")

## 4. Proyecto y tarea en CVAT

El proyecto define las etiquetas (iguales a `CLASS_NAMES`); cada lote es una tarea dividida en jobs de `SEGMENT_SIZE` imágenes.

In [ ]:
from cvat_sdk import models

indice = pd.read_csv(INDICE_FILE, dtype={"camera": str})

project = next((p for p in client.projects.list() if p.name == CVAT_PROJECT_NAME), None)
if project is None:
    project = client.projects.create(models.ProjectWriteRequest(
        name=CVAT_PROJECT_NAME,
        labels=[models.PatchedLabelRequest(name=name, type="rectangle") for name in CLASS_NAMES],
    ))
    print(f"Proyecto creado: {project.id}")
else:
    missing = set(CLASS_NAMES) - {label.name for label in project.get_labels()}
    if missing:
        raise ValueError(f"El proyecto CVAT no tiene las etiquetas {missing}: añádelas en la UI o crea otro proyecto.")
    print(f"Proyecto existente: {project.id}")

if TASK_FILE.exists():
    task = client.tasks.retrieve(json.loads(TASK_FILE.read_text())["task_id"])
    print(f"Tarea existente: {task.id}")
else:
    task = client.tasks.create_from_data(
        spec=models.TaskWriteRequest(name=f"{LOTE}", project_id=project.id, segment_size=SEGMENT_SIZE),
        resources=[str(LOTE_DIR / "images" / name) for name in indice["image"]],
        data_params={"image_quality": 95, "sorting_method": "natural"},
    )
    TASK_FILE.write_text(json.dumps({"task_id": task.id, "project_id": project.id}, indent=2))
    print(f"Tarea creada: {task.id} ({task.size} imágenes)")

print(f"Abrir en el navegador: {CVAT['CVAT_URL']}/tasks/{task.id}")

## 5. Pre-anotación automática con YOLO26

`cvat_sdk.auto_annotation.annotate_task` descarga cada imagen de la tarea, ejecuta `Yolo26PreAnnotator` (pesos COCO) en local y sube las cajas a CVAT con `source = auto`:

- `person`, `car`, `bus`, `truck`, `motorcycle`: clase COCO directa.
- Semáforos: color HSV por caja → `traffic_light_red/yellow/green`, o `traffic_light_unknown`.

Las pre-anotaciones son solo un punto de partida: se revisan todas en el paso 6. `clear_existing=True` borra lo que hubiera en la tarea, así que **no se debe relanzar tras empezar la revisión**.

In [ ]:
import torch
import cvat_sdk.auto_annotation as cvataa
from cvat_sdk.core.helpers import DeferredTqdmProgressReporter

from cvat_yolo26 import Yolo26PreAnnotator

DEVICE = 0 if torch.cuda.is_available() else "cpu"
function = Yolo26PreAnnotator(weights=PRE_WEIGHTS, imgsz=PRE_IMGSZ, conf=PRE_CONF, device=DEVICE)

cvataa.annotate_task(
    client,
    task.id,
    function,
    clear_existing=True,
    pbar=DeferredTqdmProgressReporter(),
)
print(f"Pre-anotación completada. Revisar en {CVAT['CVAT_URL']}/tasks/{task.id}")

## 6. Revisión humana en CVAT (protocolo de anotación)

Abrir la tarea en CVAT y revisar **cada job**. Criterios comunes para que las etiquetas sean consistentes (documentarlos en la memoria):

| Situación | Acción |
|---|---|
| Caja correcta | Mantener; ajustar bordes al objeto visible. |
| Falso positivo | Borrar. |
| Objeto no detectado | Dibujar caja nueva (vehículo/persona visible > ~8 px de alto). |
| Semáforo **peatonal** | Cambiar la etiqueta a `pedestrian_light_red` / `pedestrian_light_green` (la pre-anotación no los distingue). |
| Semáforo de lado, de espaldas o sin color distinguible | `traffic_light_unknown`. |
| Color mal clasificado (reflejos, halos nocturnos) | Corregir la etiqueta de color. |
| Objeto ocluido > 70 % o cortado en el borde e irreconocible | No anotar. |
| Grupo compacto de personas | Una caja por persona distinguible. |

Al terminar cada job: **Menu → Finish the job** (estado `completed`). La celda siguiente comprueba que todos los jobs estén terminados antes de exportar.

> Recomendado para la memoria: que una segunda persona anote ~10 % de las imágenes para medir el acuerdo entre anotadores (IoU / F1 entre ambas anotaciones).

In [ ]:
jobs = task.get_jobs()
estado_jobs = pd.DataFrame([{"job": j.id, "estado": str(getattr(j.state, "value", j.state)), "etapa": str(getattr(j.stage, "value", j.stage)), "asignado": getattr(j.assignee, "username", None)} for j in jobs])
display(estado_jobs)

EXPORTAR_AUNQUE_PENDIENTE = False
pendientes = estado_jobs[estado_jobs["estado"] != "completed"]
if len(pendientes) and not EXPORTAR_AUNQUE_PENDIENTE:
    raise RuntimeError(f"Quedan {len(pendientes)} jobs sin terminar. Revisa en CVAT antes de exportar.")

## 7. Exportación de las etiquetas revisadas

Se exporta en formato YOLO y se reasignan los índices de clase **por nombre** a `CLASS_NAMES`, de modo que el orden de etiquetas en CVAT no importa. Una imagen revisada sin cajas genera un `.txt` vacío (ejemplo negativo válido).

In [ ]:
export_dir = LOTE_DIR / "export"
labels_dir = LOTE_DIR / "labels"
shutil.rmtree(export_dir, ignore_errors=True)
export_dir.mkdir(parents=True)

zip_path = task.export_dataset(EXPORT_FORMAT, export_dir / f"{LOTE}.zip", include_images=False)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(export_dir / "unzipped")
unzipped = export_dir / "unzipped"

# Nombres de clase del export y ficheros de etiquetas (src/anotacion.py)
export_names = read_export_names(unzipped)
label_files = find_label_files(unzipped)

shutil.rmtree(labels_dir, ignore_errors=True)
labels_dir.mkdir()
indice = pd.read_csv(INDICE_FILE, dtype={"camera": str})
n_cajas = []
for image in indice["image"]:
    stem = Path(image).stem
    raw_lines = label_files[stem].read_text(encoding="utf-8").splitlines() if stem in label_files else []
    lines = remap_lines(raw_lines, export_names, CLASS_TO_IDX)
    (labels_dir / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    n_cajas.append(len(lines))

indice["label_path"] = [f"labels/{Path(i).stem}.txt" for i in indice["image"]]
indice["n_cajas"] = n_cajas
indice["estado"] = "revisada"
indice.to_csv(INDICE_FILE, index=False)

print(f"Etiquetas escritas en {labels_dir} ({len(indice)} imágenes, {sum(n_cajas)} cajas)")

## 8. Resumen del lote

Distribución de cajas por clase y cámara. Clases con muy pocas instancias (p. ej. `traffic_light_yellow`) indican que el siguiente lote debe sobremuestrear esas situaciones.

In [ ]:
rows = []
for label_file in labels_dir.glob("*.txt"):
    camera = label_file.stem.split("_", 1)[0]
    for line in label_file.read_text(encoding="utf-8").splitlines():
        rows.append({"camera": camera, "clase": CLASS_NAMES[int(line.split()[0])]})

boxes = pd.DataFrame(rows, columns=["camera", "clase"])
print("Cajas por clase:")
display(boxes["clase"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_frame("cajas"))
print(f"Imágenes sin ningún objeto: {(indice['n_cajas'] == 0).sum()}")
display(pd.crosstab(boxes["camera"], boxes["clase"]))

## Siguiente fase

`04_training.ipynb` lee todos los `data/malaga/anotacion/*/indice.csv` con `estado = revisada`, junto con sus `labels/`, y entrena YOLO26 con validación cruzada por cámara.